# Entity Recommendations

## Goal

Given an input clinical study, recommend entities commonly used in similar studies but missing from the input study.

## "Dumb" vs "Smart" Recommendations

**Dumb** recommendations use co-occurrence alone. **Smart** recommendations keep that evidence and rerank it with the deviation score and the specialized network score so common hub entities fall behind entities that are specific to the neighborhood.


In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.recommendation_engine import RecommendationEngine
from src.similarity import protocol_studies
import pandas as pd
target_study = protocol_studies()["study_name"].tolist()[0]
engine = RecommendationEngine(similarity_top_k=5, cooccurrence_min_pct=0, cooccurrence_min_count=1)


## Find Top K Similar Studies

In [2]:

result = engine.recommend(target_study)
display(pd.DataFrame([{"study_id": study_id, "similarity": item.study_design_similarity} for study_id, item in result["similar_studies"]]))


,study_id,similarity
0,NCT02427958,0.664822
1,NCT02879578,0.645054
2,NCT02709655,0.618838
3,NCT02188121,0.616638
4,NCT03461016,0.573085


## Smart Recommendations

In [3]:

display(result["aggregated"].head(15))
if not result["cooccurrence"].empty:
    ranked = result["cooccurrence"].copy()
    from src.recommendations import rerank_specialized_network
    ranked = rerank_specialized_network(ranked, result["target_entities"])
    display(ranked.head(10))


,Recommended Entity,Entity type,Frequency,Example,Natural language
0,INCLUSION_CRITERION- Age requirement,INCLUSION_CRITERION,2,OBJECTIVE- Assess long-term safety,Studies that used OBJECTIVE- Assess long-term ...
1,ASSESSMENT- Acceptability rating of mobile phone,ASSESSMENT,1,INCLUSION_CRITERION- Age 5 years or older,Studies that used INCLUSION_CRITERION- Age 5 y...
2,ASSESSMENT- AARP assessment,ASSESSMENT,1,INCLUSION_CRITERION- Age 5 years or older,Studies that used INCLUSION_CRITERION- Age 5 y...
3,ASSESSMENT- Children’s Global Assessment Scale,ASSESSMENT,1,OBJECTIVE- Assess long-term safety,Studies that used OBJECTIVE- Assess long-term ...
4,ASSESSMENT- Columbia-Suicide Severity Rating S...,ASSESSMENT,1,OBJECTIVE- Assess long-term safety,Studies that used OBJECTIVE- Assess long-term ...
5,ASSESSMENT- Comparison of statin doses,ASSESSMENT,1,INCLUSION_CRITERION- Age 5 years or older,Studies that used INCLUSION_CRITERION- Age 5 y...
6,ASSESSMENT- Children Depression Rating Scale R...,ASSESSMENT,1,OBJECTIVE- Assess long-term safety,Studies that used OBJECTIVE- Assess long-term ...
7,ASSESSMENT- Control group DFS assessment,ASSESSMENT,1,INCLUSION_CRITERION- Age 5 years or older,Studies that used INCLUSION_CRITERION- Age 5 y...
8,ASSESSMENT- DBAT,ASSESSMENT,1,INCLUSION_CRITERION- Age 5 years or older,Studies that used INCLUSION_CRITERION- Age 5 y...
9,ASSESSMENT- Demographic questionnaire,ASSESSMENT,1,INCLUSION_CRITERION- Age 5 years or older,Studies that used INCLUSION_CRITERION- Age 5 y...


,Target Entity,Recommended Entity,Co-occurrence Count,Target Entity Studies,Recommended Entity Studies,Co-occurrence %,Co-occurring Studies,entity_in_target,entity_suggested,neighbor_studies_with_pair,Natural language,pagerank_suggested,betweenness_suggested,target_overlap,network_score,hub_adjustment,specialized_network_score
0,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Efficacy and safety analysis,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
1,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ASSESSMENT- Overdose documentation,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
2,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- BMI measurement,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
3,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in bone mineral density,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
4,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Change in laboratory values,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
5,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Decrease in BA to chronological age ...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
6,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Improvement in predicted adult height,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
7,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Incidence of polycystic ovarian synd...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
8,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Regression or no progression in Tann...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
9,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,1,1,100.0,NCT02427958,INCLUSION_CRITERION- Written informed consent,ENDPOINT- Suppression of basal estradiol or te...,1,Studies that used INCLUSION_CRITERION- Written...,0.005668,0.000461,3,1.505783,0.988971,1.489175
